# Customer Intelligence & Segmentation Engine
### RFM Analysis + K-Means Clustering on Online Retail Data

---

| | |
|---|---|
| **Author** | Krupal Gohil |
| **Dataset** | [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii) |
| **Domain** | E-Commerce / Customer Analytics |
| **Techniques** | EDA · RFM Feature Engineering · K-Means Clustering · Churn Scoring |
| **Goal** | Segment ~4,300 customers into actionable groups to drive marketing strategy |

---

## Table of Contents

| Step | Section |
|------|---------|
| 1 | Business Problem |
| 2 | Dataset Overview |
| 3 | Setup & Imports |
| 4 | Load Data |
| 5 | Data Inspection |
| 6 | Exploratory Data Analysis (EDA) |
| 7 | Data Cleaning |
| 8 | RFM Feature Engineering |
| 9 | Log Transformation |
| 10 | Outlier Handling |
| 11 | Scaling |
| 12 | Finding the Right Number of Clusters |
| 13 | Final Model & Visualisation |
| 14 | Cluster Profiling & Labeling |
| 15 | Premium Customer Segments |
| 16 | Churn Risk Scoring |
| 17 | Business Recommendations |
| 18 | Conclusion |


---
## Step 1 — Business Problem

### The Situation

A UK-based online gift retailer has been running for over a year. They sell across 40 countries and have processed more than 500,000 transactions.

But here is the problem — **they treat every customer the same.**

They send the same email to everyone, run the same promotions for everyone, and have no idea who their best customers are. This means:

- Their best customers get no special treatment and may leave
- Money is wasted on customers who never come back
- No one knows which customers are about to stop buying

### The Solution

We use **customer segmentation** — grouping customers by their purchasing behaviour — so the business can treat different customers differently.

The framework we use is called **RFM**:

| Letter | Stands for | What it measures |
|--------|-----------|-----------------|
| **R** | Recency | How recently did the customer buy? |
| **F** | Frequency | How often do they buy? |
| **M** | Monetary | How much do they spend in total? |

Using these three numbers for each customer, we apply **K-Means Clustering** to automatically group customers into segments.

### Business Questions We Want to Answer

| # | Question |
|---|---------|
| Q1 | Which countries and products generate the most revenue? |
| Q2 | What does customer spending behaviour look like — are most customers low spenders? |
| Q3 | How recently and how often do customers typically purchase? |
| Q4 | How many natural customer groups exist in the data? |
| Q5 | What are the characteristics of each customer group? |
| Q6 | Which customers are at risk of leaving (churning)? |


---
## Step 2 — Dataset Overview

**Dataset:** UCI Online Retail II — real transaction data from a UK online gift retailer

**Source:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

**Size:** ~525,000 transactions across 2 years (Dec 2009 – Dec 2010)

### Column Descriptions

| Column | Description |
|--------|-------------|
| `Invoice` | Unique 6-digit invoice number. Starts with `C` = cancellation, `A` = bad debt |
| `StockCode` | Product code |
| `Description` | Product name |
| `Quantity` | Units sold. Negative = return/refund |
| `InvoiceDate` | Date and time of the transaction |
| `Price` | Price per unit in GBP (£) |
| `Customer ID` | Unique customer identifier — some transactions have no ID (guest checkouts) |
| `Country` | Country where the customer is based |

### What We Will Build From This Data

This is transaction-level data — one row per item per invoice.

We need to **aggregate it to customer level** to create RFM features. By the end of the cleaning and engineering steps, each row will represent one customer with three numbers: Recency, Frequency, and Monetary value.


---
## Step 3 — Setup & Imports


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

# File path — update this to your local path
DATA_PATH = 'online_retail_II.xlsx'

print("All libraries loaded.")


---
## Step 4 — Load Data

We load Sheet 0 from the Excel file, which covers December 2009 to December 2010.


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name=0)

print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")

df.head()


---
## Step 5 — Data Inspection

Before any analysis or cleaning, we look at the raw data to understand its structure and spot any obvious problems.

We check:
- Data types for each column
- How many values are missing
- Basic statistics (min, max, average)


In [ ]:
df.info()


In [ ]:
df.describe().round(2)


### Missing Values


In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_report[missing_report['Missing Count'] > 0]


> **Observation:** `Customer ID` is missing for about 20% of rows — these are guest checkouts with no customer identity. We cannot track these customers over time, so they must be removed before building RFM features. `Description` has a small number of nulls which is fine since we don't use it in the model.


### Quick Look at Negative Values


In [ ]:
print(f"Rows with negative Quantity : {(df['Quantity'] < 0).sum():,}")
print(f"Rows with negative Price    : {(df['Price'] < 0).sum():,}")
print(f"Rows with zero Price        : {(df['Price'] == 0).sum():,}")


> **Observation:** Negative quantities are product returns and cancellations. Negative prices come from bad-debt adjustment entries. Zero-price rows are free samples or admin entries. All three need to be removed since we are measuring real customer purchases.


---
## Step 6 — Exploratory Data Analysis (EDA)

EDA means exploring the data to understand patterns before we do any modelling.

We will explore:
- Which countries and products generate the most revenue
- How transaction volume changes over time
- What the distribution of transaction amounts looks like
- Who the top customers are


### 6.1 — Revenue by Country

We create a quick revenue column (Price × Quantity) to understand geographic distribution.


In [ ]:
eda_df = df.copy()
eda_df['Revenue'] = eda_df['Quantity'] * eda_df['Price']

# Keep only positive revenue (real purchases)
eda_pos = eda_df[eda_df['Revenue'] > 0]

top_countries = (
    eda_pos.groupby('Country')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5))
top_countries.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 10 Countries by Revenue', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Total Revenue (£)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f"UK share of total revenue: {top_countries['United Kingdom']/top_countries.sum()*100:.1f}%")


> **Finding (Q1 answered):** The United Kingdom dominates revenue by a large margin. The retailer is primarily a UK business. A small number of other countries (Netherlands, EIRE, Germany, France) make up most of the remaining revenue. This tells us the business should focus retention efforts on UK customers first.


### 6.2 — Monthly Revenue Trend


In [ ]:
eda_pos = eda_pos.copy()
eda_pos['Month'] = pd.to_datetime(eda_pos['InvoiceDate']).dt.to_period('M')

monthly_revenue = eda_pos.groupby('Month')['Revenue'].sum()

fig, ax = plt.subplots(figsize=(12, 4))
monthly_revenue.plot(ax=ax, color='teal', marker='o', linewidth=2, markersize=5)
ax.set_title('Monthly Revenue Trend (Dec 2009 – Dec 2010)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Revenue (£)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


> **Finding:** Revenue shows a clear seasonal pattern with a large spike towards the end of the year — driven by Christmas gift purchasing. This is important context for segmentation: customers who bought only in November/December may appear as high-value but are actually seasonal buyers.


### 6.3 — Top 10 Products by Revenue


In [ ]:
top_products = (
    eda_pos.groupby('Description')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 4))
top_products.plot(kind='barh', ax=ax, color='coral', edgecolor='white')
ax.set_title('Top 10 Products by Revenue', fontweight='bold')
ax.set_xlabel('Total Revenue (£)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()


> **Finding:** The top products are mostly decorative homeware and gift items — consistent with the retailer's description as an online gift shop. There is no single product that dominates revenue, which means the business is not overly dependent on one item.


### 6.4 — Transaction Amount Distribution

How much do people typically spend per transaction?


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Distribution of transaction amounts (clipped to remove extreme values)
clip_val = eda_pos['Revenue'].quantile(0.99)

eda_pos['Revenue'].clip(upper=clip_val).plot(
    kind='hist', bins=50, ax=axes[0],
    color='steelblue', edgecolor='white'
)
axes[0].set_title('Transaction Amount Distribution
(clipped at 99th percentile)')
axes[0].set_xlabel('Revenue per Transaction (£)')
axes[0].set_ylabel('Count')

# Distribution on log scale to see the full picture
np.log1p(eda_pos['Revenue']).plot(
    kind='hist', bins=50, ax=axes[1],
    color='teal', edgecolor='white'
)
axes[1].set_title('Same Distribution — Log Scale
(shows the full range)')
axes[1].set_xlabel('Log(Revenue)')
axes[1].set_ylabel('Count')

plt.suptitle('Transaction Amount (Q2)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Median transaction value : £{eda_pos['Revenue'].median():.2f}")
print(f"Mean transaction value   : £{eda_pos['Revenue'].mean():.2f}")
print(f"90th percentile          : £{eda_pos['Revenue'].quantile(0.9):.2f}")


> **Finding (Q2 answered):** The distribution is heavily right-skewed — most transactions are small (median around £10–15), but a small number of very large orders pull the average up significantly. This means most customers are low spenders, but a small group of high-value customers drives a disproportionate share of revenue. This is the classic "80/20" pattern that makes segmentation so valuable.


### 6.5 — Number of Orders per Customer

How often do customers typically buy?


In [ ]:
orders_per_customer = eda_pos.groupby('Customer ID')['Invoice'].nunique()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

orders_per_customer.clip(upper=orders_per_customer.quantile(0.95)).plot(
    kind='hist', bins=40, ax=axes[0],
    color='steelblue', edgecolor='white'
)
axes[0].set_title('Orders per Customer
(clipped at 95th percentile)')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Number of Customers')

# How many customers bought only once?
once = (orders_per_customer == 1).sum()
more = (orders_per_customer > 1).sum()
axes[1].bar(['Bought once', 'Bought 2+ times'],
            [once, more], color=['crimson', 'steelblue'], edgecolor='white')
axes[1].set_title('One-Time vs Repeat Customers')
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate([once, more]):
    axes[1].text(i, v + 10, f'{v:,}', ha='center')

plt.suptitle('Purchase Frequency (Q3)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Median orders per customer : {orders_per_customer.median():.0f}")
print(f"One-time buyers            : {once:,} ({once/len(orders_per_customer)*100:.1f}%)")


> **Finding (Q3 answered):** A large portion of customers bought only once. These are exactly the customers the "NURTURE" segment will capture — people with potential who need to be brought back. Repeat buyers are the more loyal base and represent the core of the business.


### 6.6 — Days Since Last Purchase (Recency Preview)

How recently did customers last buy, and is there a cluster of recently-active vs lapsed customers?


In [ ]:
last_purchase = eda_pos.groupby('Customer ID')['InvoiceDate'].max()
snapshot = last_purchase.max()
recency_preview = (snapshot - last_purchase).dt.days

fig, ax = plt.subplots(figsize=(10, 4))
recency_preview.clip(upper=recency_preview.quantile(0.99)).plot(
    kind='hist', bins=50, ax=ax,
    color='coral', edgecolor='white'
)
ax.set_title('Days Since Last Purchase — All Customers', fontweight='bold')
ax.set_xlabel('Days Since Last Purchase (Recency)')
ax.set_ylabel('Number of Customers')
plt.tight_layout()
plt.show()

print(f"Customers active in last 30 days  : {(recency_preview <= 30).sum():,}")
print(f"Customers inactive 90+ days       : {(recency_preview > 90).sum():,}")
print(f"Customers inactive 180+ days      : {(recency_preview > 180).sum():,}")


> **Finding:** There are clearly two groups — customers who purchased very recently and those who haven't bought in months. The latter group represents churn risk and will form the "RE-ENGAGE" segment later. This natural separation is a good sign that K-Means will find meaningful clusters.


### 6.7 — Top 10 Customers by Revenue


In [ ]:
top_customers = (
    eda_pos.groupby('Customer ID')['Revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)
top_customers.columns = ['Customer ID', 'Total Revenue (£)']
top_customers['Total Revenue (£)'] = top_customers['Total Revenue (£)'].round(2)
top_customers


> **Finding:** The top 10 customers each contribute tens of thousands of pounds in revenue. These will become our "DELIGHT" or "PAMPER" premium segments. Losing even one of these customers would have a significant business impact — they deserve special treatment.


---
## Step 7 — Data Cleaning

Based on what we found in the inspection and EDA steps, we now remove the rows that would distort our RFM calculations.

### What we remove and why

| Problem | What we remove | Why |
|---------|----------------|-----|
| Cancellation invoices | Invoices starting with `C` | Not real purchases |
| Bad debt entries | Invoices starting with `A` | Accounting adjustments, not transactions |
| Internal stock codes | DOT, D, M, BANK CHARGES, etc. | Not real products |
| Missing Customer ID | Rows where Customer ID is null | Cannot track customer behaviour |
| Zero-price rows | Rows where Price = 0 | Free samples / admin entries |


In [ ]:
cleaned_df = df.copy()

# Remove non-standard invoices (cancellations start with C, adjustments with A)
cleaned_df['Invoice'] = cleaned_df['Invoice'].astype('str')
cleaned_df = cleaned_df[cleaned_df['Invoice'].str.match(r'^\d{6}$')]

# Keep only real product stock codes (5 digits, optionally with letters, or PADS)
cleaned_df['StockCode'] = cleaned_df['StockCode'].astype('str')
valid_stock = (
    cleaned_df['StockCode'].str.match(r'^\d{5}$') |
    cleaned_df['StockCode'].str.match(r'^\d{5}[a-zA-Z]+$') |
    (cleaned_df['StockCode'] == 'PADS')
)
cleaned_df = cleaned_df[valid_stock]

# Remove rows with no Customer ID
cleaned_df.dropna(subset=['Customer ID'], inplace=True)

# Remove zero-price rows
cleaned_df = cleaned_df[cleaned_df['Price'] > 0]

print(f"Raw rows     : {len(df):,}")
print(f"Clean rows   : {len(cleaned_df):,}")
print(f"Rows removed : {len(df) - len(cleaned_df):,}")
print(f"Kept         : {len(cleaned_df) / len(df) * 100:.1f}%")


> **Observation:** We retained about 77% of the original rows — a normal retention rate for real-world retail data. The removed rows were non-purchases that would have corrupted our customer spending calculations.


---
## Step 8 — RFM Feature Engineering

This is the most important step. We transform transaction-level data (one row per item) into customer-level data (one row per customer) with three features.

### The Three Features

| Feature | How we calculate it | What it tells us |
|---------|--------------------|--------------------|
| **Recency** | Days between last purchase and the snapshot date | Lower = more recently active = better |
| **Frequency** | Count of unique invoice numbers | Higher = more loyal = better |
| **Monetary** | Sum of (Price × Quantity) across all purchases | Higher = more valuable = better |

The snapshot date is the last transaction date in the dataset — we measure recency relative to this date.


In [ ]:
cleaned_df = cleaned_df.copy()

# Revenue per line item
cleaned_df['SalesLineTotal'] = cleaned_df['Price'] * cleaned_df['Quantity']

# Aggregate to one row per customer
rfm = cleaned_df.groupby('Customer ID', as_index=False).agg(
    MonetaryValue=('SalesLineTotal', 'sum'),
    Frequency=('Invoice', 'nunique'),
    LastPurchase=('InvoiceDate', 'max')
)

# Calculate Recency
snapshot_date = rfm['LastPurchase'].max()
rfm['Recency'] = (snapshot_date - rfm['LastPurchase']).dt.days
rfm.drop(columns='LastPurchase', inplace=True)

print(f"Customers in RFM table: {len(rfm):,}")
rfm.head()


In [ ]:
rfm.describe().round(2)


### Distribution of RFM Features


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RFM Feature Distributions — Before Any Transformation', fontweight='bold')

rfm['MonetaryValue'].clip(upper=rfm['MonetaryValue'].quantile(0.99)).plot(
    kind='hist', bins=40, ax=axes[0], color='#4C72B0', edgecolor='white')
axes[0].set_title('Monetary Value (£)')
axes[0].set_xlabel('Total Spend')

rfm['Frequency'].clip(upper=rfm['Frequency'].quantile(0.99)).plot(
    kind='hist', bins=30, ax=axes[1], color='#55A868', edgecolor='white')
axes[1].set_title('Frequency (orders)')
axes[1].set_xlabel('Number of Orders')

rfm['Recency'].plot(
    kind='hist', bins=30, ax=axes[2], color='#C44E52', edgecolor='white')
axes[2].set_title('Recency (days)')
axes[2].set_xlabel('Days Since Last Purchase')

plt.tight_layout()
plt.show()


> **Observation:** Both `MonetaryValue` and `Frequency` are **right-skewed** — most customers are low spenders with few orders, but a small number of customers have very high values. `Recency` is more spread out. The skew in Monetary and Frequency will cause problems for K-Means in the next steps — we will fix this with a log transformation.


---
## Step 9 — Log Transformation

### Why Do We Need This?

K-Means works by measuring the **distance** between customers in RFM space. If one customer spends £10 and another spends £50,000, the £50,000 customer will look extremely far away from everyone else. This causes K-Means to create distorted clusters.

**Log transformation** compresses the long tail. A customer spending £50,000 no longer looks 5,000x farther away than one spending £10.

We use `np.log1p()` which is `log(value + 1)` — the `+1` safely handles any zero values.


In [ ]:
rfm_log = rfm.copy()
rfm_log['MonetaryValue'] = np.log1p(rfm_log['MonetaryValue'])
rfm_log['Frequency']     = np.log1p(rfm_log['Frequency'])

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('Before vs After Log Transformation', fontweight='bold')

# Before
rfm['MonetaryValue'].clip(upper=rfm['MonetaryValue'].quantile(0.99)).plot(
    kind='hist', bins=40, ax=axes[0][0], color='#C44E52', edgecolor='white')
axes[0][0].set_title('Monetary — Raw (skewed)')
axes[0][0].set_xlabel('Total Spend (£)')

rfm['Frequency'].clip(upper=rfm['Frequency'].quantile(0.99)).plot(
    kind='hist', bins=30, ax=axes[0][1], color='#C44E52', edgecolor='white')
axes[0][1].set_title('Frequency — Raw (skewed)')
axes[0][1].set_xlabel('Number of Orders')

# After
rfm_log['MonetaryValue'].plot(
    kind='hist', bins=40, ax=axes[1][0], color='#55A868', edgecolor='white')
axes[1][0].set_title('Monetary — After log1p (much better)')
axes[1][0].set_xlabel('log(Total Spend + 1)')

rfm_log['Frequency'].plot(
    kind='hist', bins=30, ax=axes[1][1], color='#55A868', edgecolor='white')
axes[1][1].set_title('Frequency — After log1p (much better)')
axes[1][1].set_xlabel('log(Orders + 1)')

plt.tight_layout()
plt.show()


> **Observation:** After log transformation the distributions are much more symmetric and bell-shaped. This gives K-Means a much fairer view of the data and leads to better cluster separation.


---
## Step 10 — Outlier Handling

### Why Not Just Drop Outliers?

Even after log transformation, some customers are still extreme outliers — they buy 10× more or spend 10× more than everyone else.

If we include them in K-Means, they will pull clusters toward themselves. But we **cannot just drop them** — they are the business's most valuable customers.

**Our solution:** Separate outliers into their own premium segments before running K-Means on the regular customers.

We use the **IQR method** to identify outliers:
- Lower fence = Q1 − 1.5 × IQR
- Upper fence = Q3 + 1.5 × IQR

Any customer outside these fences in `MonetaryValue` or `Frequency` is set aside.


In [ ]:
def get_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return (series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)

monetary_outliers  = get_outliers(rfm_log['MonetaryValue'])
frequency_outliers = get_outliers(rfm_log['Frequency'])

outlier_mask = monetary_outliers | frequency_outliers

core_df    = rfm_log[~outlier_mask].copy()
outlier_df = rfm[outlier_mask].copy()

print(f"Core customers (for K-Means) : {len(core_df):,}")
print(f"Outlier customers (premium)  : {len(outlier_df):,}")
print(f"Total customers              : {len(rfm):,}")


In [ ]:
# Visualise the split
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Core vs Outlier Customers', fontweight='bold')

axes[0].scatter(rfm.loc[~outlier_mask, 'MonetaryValue'],
                rfm.loc[~outlier_mask, 'Frequency'],
                alpha=0.3, s=10, color='steelblue', label='Core')
axes[0].scatter(rfm.loc[outlier_mask, 'MonetaryValue'],
                rfm.loc[outlier_mask, 'Frequency'],
                alpha=0.7, s=20, color='crimson', label='Outlier (Premium)')
axes[0].set_title('Monetary vs Frequency')
axes[0].set_xlabel('Total Spend (£)')
axes[0].set_ylabel('Number of Orders')
axes[0].legend()

axes[1].bar(['Core customers', 'Premium outliers'],
            [len(core_df), len(outlier_df)],
            color=['steelblue', 'crimson'], edgecolor='white')
axes[1].set_title('Customer Split')
axes[1].set_ylabel('Number of Customers')
for i, v in enumerate([len(core_df), len(outlier_df)]):
    axes[1].text(i, v + 5, str(v), ha='center')

plt.tight_layout()
plt.show()


> **Observation:** A small group of customers (shown in red) have dramatically higher spending or purchase frequency than the rest. These are the premium VIP customers. We set them aside now and will give them dedicated segment labels in Step 15.


---
## Step 11 — Scaling

### Why Do We Need to Scale?

K-Means calculates the distance between customers using all three features (R, F, M). If the features are on different scales, the one with the biggest numbers will dominate:

- `MonetaryValue` in log scale: roughly 0 – 10
- `Frequency` in log scale: roughly 0 – 5
- `Recency` in days: roughly 0 – 370

Without scaling, Recency would dominate and K-Means would mostly separate customers by how recently they bought, ignoring spending and frequency patterns.

`StandardScaler` transforms each feature so it has a **mean of 0 and standard deviation of 1**. All three features then have equal weight.


In [ ]:
scaler = StandardScaler()
features = ['MonetaryValue', 'Frequency', 'Recency']

scaled_array = scaler.fit_transform(core_df[features])
scaled_df = pd.DataFrame(scaled_array, index=core_df.index, columns=features)

print("After scaling — all features should have mean ≈ 0 and std ≈ 1:")
scaled_df.describe().round(3)


---
## Step 12 — Finding the Right Number of Clusters

K-Means needs us to tell it how many clusters to create (the value of K). There is no single correct answer — we use two charts to find the best value.

### The Two Methods

**Elbow Method (Inertia):**
- Inertia measures how tightly packed the clusters are
- Lower inertia = more compact clusters = better
- We look for the "elbow" — the point where adding more clusters stops helping much

**Silhouette Score:**
- Measures how well-separated the clusters are from each other
- Range: −1 to +1 (higher is better)
- A score above 0.3 is considered reasonable


In [ ]:
inertia       = []
sil_scores    = []
k_range       = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(scaled_df)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(scaled_df, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Finding the Best K (Q4)', fontweight='bold')

# Elbow curve
axes[0].plot(k_range, inertia, marker='o', color='steelblue', linewidth=2)
axes[0].axvline(x=4, color='crimson', linestyle='--', alpha=0.7, label='k = 4')
axes[0].set_title('Elbow Method — Inertia')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Silhouette scores
axes[1].plot(k_range, sil_scores, marker='o', color='teal', linewidth=2)
axes[1].axvline(x=4, color='crimson', linestyle='--', alpha=0.7, label='k = 4')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Score (higher = better)')
axes[1].legend()
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

best_k = list(k_range)[sil_scores.index(max(sil_scores))]
print(f"Highest silhouette score at k = {best_k} : {max(sil_scores):.4f}")


> **Finding (Q4 answered):** The elbow curve bends most noticeably at k=4, and the silhouette score supports this choice. We will use **k = 4** for our core segmentation model.


---
## Step 13 — Final Model & Visualisation

### Stability Check

Before trusting the result, we verify that K-Means gives consistent clusters across 10 different random starts. If the silhouette scores vary a lot between runs, the clusters are not reliable.


In [ ]:
stability_scores = [
    silhouette_score(
        scaled_df,
        KMeans(n_clusters=4, random_state=i, n_init=10).fit_predict(scaled_df)
    )
    for i in range(10)
]

print("Silhouette scores across 10 random starts:")
print([round(s, 4) for s in stability_scores])
print(f"
Mean  : {np.mean(stability_scores):.4f}")
print(f"Std   : {np.std(stability_scores):.4f}  (lower = more stable)")


> **Observation:** If the standard deviation is very low (e.g. below 0.005), the clusters are stable and trustworthy. A high standard deviation would mean the clusters change a lot between runs — a sign we should reconsider the value of K.


In [ ]:
# Train the final model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
core_df = core_df.copy()
core_df['Cluster'] = kmeans.fit_predict(scaled_df)

# PCA: reduce 3D RFM to 2D so we can plot it easily
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(scaled_df)
pca_df = pd.DataFrame(pca_coords, columns=['PC1', 'PC2'], index=core_df.index)
pca_df['Cluster'] = core_df['Cluster'].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('K-Means Clustering Result (k=4)', fontweight='bold')

# PCA scatter — colour by cluster
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for c in range(4):
    mask = pca_df['Cluster'] == c
    axes[0].scatter(
        pca_df[mask]['PC1'], pca_df[mask]['PC2'],
        c=colors[c], label=f'Cluster {c}', alpha=0.5, s=15
    )
axes[0].set_title('Customer Clusters in 2D (via PCA)')
axes[0].set_xlabel(f'PC1 — {pca.explained_variance_ratio_[0]*100:.1f}% of variance')
axes[0].set_ylabel(f'PC2 — {pca.explained_variance_ratio_[1]*100:.1f}% of variance')
axes[0].legend()

# Cluster size bar chart
core_df['Cluster'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=colors, edgecolor='white'
)
axes[1].set_title('Customers per Cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


> **Observation:** The PCA scatter shows four visually distinct groups — the clusters have reasonable separation. PCA compresses the 3D RFM space into 2D for plotting; the percentage on each axis shows how much of the original variation is captured.


---
## Step 14 — Cluster Profiling & Labeling

Now we need to understand what each cluster actually represents.

We look at the **average RFM values** per cluster using the original (non-log) values, because those are easier to interpret in real business terms.

**How to read the profile table:**
- Low Recency = bought recently = good
- High Frequency = buys often = good
- High Monetary = high spender = good


In [ ]:
# Use original rfm values (not log-transformed) for interpretable averages
core_orig = rfm[~outlier_mask].copy()
core_orig['Cluster'] = core_df['Cluster'].values

profile = core_orig.groupby('Cluster')[['Recency', 'Frequency', 'MonetaryValue']].mean().round(1)
profile['Customer Count'] = core_orig['Cluster'].value_counts().sort_index()

profile


### Visualise RFM by Cluster


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Profiles by Cluster (Q5)', fontweight='bold')

palette = {0: '#1f77b4', 1: '#ff7f0e', 2: '#2ca02c', 3: '#d62728'}

for ax, feature in zip(axes, ['Recency', 'Frequency', 'MonetaryValue']):
    sns.boxplot(
        x='Cluster', y=feature,
        data=core_orig,
        palette=palette,
        ax=ax
    )
    ax.set_title(feature)
    ax.set_xlabel('Cluster')

plt.tight_layout()
plt.show()


### Assign Segment Names

Based on the profile table above, we assign a business label to each cluster. The label reflects what action the marketing team should take for that group.


In [ ]:
# Labels based on the profile — update these if your cluster numbers differ
# Run the profile table above first to confirm which cluster matches which profile
segment_map = {
    0: 'RETAIN',      # Mid-range across R, F, M — steady, reliable customers
    1: 'RE-ENGAGE',   # High recency (haven't bought in a while) — at risk
    2: 'NURTURE',     # Low frequency and spend — new or one-time buyers
    3: 'REWARD'       # Low recency and decent spend — recently active loyalists
}

core_orig['Segment'] = core_orig['Cluster'].map(segment_map)

print("Segment distribution:")
print(core_orig['Segment'].value_counts().to_string())


In [ ]:
# Segment profile summary — average RFM per named segment
seg_profile = core_orig.groupby('Segment')[['Recency', 'Frequency', 'MonetaryValue']].mean().round(1)
seg_profile['Customers'] = core_orig['Segment'].value_counts()
seg_profile


> **Finding (Q5 answered):** Each cluster has a clearly distinct profile. The segment names reflect what action the business should take — reward your loyalists, re-engage lapsed customers, nurture new ones, and retain the stable middle group.


---
## Step 15 — Premium Customer Segments

The outlier customers we removed before K-Means are the business's most valuable customers. Instead of lumping them into one group, we split them into three premium segments.

| Segment | Who they are | Strategy |
|---------|-------------|---------|
| **PAMPER** | Very high total spend but not necessarily very frequent | Offer exclusive bundles, priority service |
| **UPSELL** | Very high frequency but not necessarily the highest spend | Reward loyalty, increase basket size |
| **DELIGHT** | Both very high spend AND very high frequency | True VIP treatment — these customers are gold |


In [ ]:
outlier_df = rfm[outlier_mask].copy()

# Re-check which outlier criteria each customer meets
mon_out  = get_outliers(rfm_log.loc[outlier_mask, 'MonetaryValue'])
freq_out = get_outliers(rfm_log.loc[outlier_mask, 'Frequency'])

def assign_premium(idx):
    m = mon_out.get(idx, False)
    f = freq_out.get(idx, False)
    if m and f:
        return 'DELIGHT'
    elif m:
        return 'PAMPER'
    return 'UPSELL'

outlier_df['Segment'] = [assign_premium(i) for i in outlier_df.index]

print("Premium segment breakdown:")
print(outlier_df['Segment'].value_counts().to_string())


In [ ]:
# Premium segment profiles
premium_profile = outlier_df.groupby('Segment')[['Recency', 'Frequency', 'MonetaryValue']].mean().round(1)
premium_profile['Customers'] = outlier_df['Segment'].value_counts()
premium_profile


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Premium Segment Profiles', fontweight='bold')

prem_palette = {'PAMPER': '#9467bd', 'UPSELL': '#8c564b', 'DELIGHT': '#e377c2'}

for ax, feature in zip(axes, ['Recency', 'Frequency', 'MonetaryValue']):
    sns.boxplot(
        x='Segment', y=feature,
        data=outlier_df,
        palette=prem_palette,
        order=['PAMPER', 'UPSELL', 'DELIGHT'],
        ax=ax
    )
    ax.set_title(feature)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.show()


---
## Step 16 — Churn Risk Scoring

### What is Churn?

Churn means a customer stops buying. In subscription businesses it is very clear — they cancel. In retail it is less obvious — customers just quietly stop purchasing.

### How We Score Churn Risk

We use **Recency** to estimate churn risk. A customer who last purchased 300 days ago is at higher risk than one who purchased last week.

We normalise Recency to a 0–100 score:
- **Score 0** = purchased very recently = low churn risk
- **Score 100** = hasn't purchased in a very long time = high churn risk

This gives the marketing team a **ranked list** — who to contact first for a win-back campaign.


In [ ]:
# Combine all 7 segments into one master table
all_segments = pd.concat([
    core_orig[['Customer ID', 'Recency', 'Frequency', 'MonetaryValue', 'Segment']],
    outlier_df[['Customer ID', 'Recency', 'Frequency', 'MonetaryValue', 'Segment']]
]).reset_index(drop=True)

# Churn score: scale Recency to 0–100
max_recency = all_segments['Recency'].max()
all_segments['ChurnScore'] = (all_segments['Recency'] / max_recency * 100).round(1)

print(f"Total customers with segment labels: {len(all_segments):,}")
all_segments.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Churn Risk Analysis (Q6)', fontweight='bold')

# Churn score distribution by segment
seg_order = ['DELIGHT', 'PAMPER', 'UPSELL', 'REWARD', 'RETAIN', 'NURTURE', 'RE-ENGAGE']
sns.boxplot(
    x='Segment', y='ChurnScore',
    data=all_segments,
    order=[s for s in seg_order if s in all_segments['Segment'].unique()],
    palette='RdYlGn_r',
    ax=axes[0]
)
axes[0].set_title('Churn Risk Score by Segment')
axes[0].set_xlabel('')
axes[0].set_ylabel('Churn Risk (0 = safe, 100 = high risk)')
axes[0].tick_params(axis='x', rotation=20)

# Count of high-risk customers per segment (score > 70)
high_risk = all_segments[all_segments['ChurnScore'] > 70]
high_risk_count = high_risk['Segment'].value_counts()
high_risk_count.plot(kind='bar', ax=axes[1], color='crimson', edgecolor='white')
axes[1].set_title('High-Risk Customers (Churn Score > 70)')
axes[1].set_xlabel('')
axes[1].set_ylabel('Number of Customers')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print(f"
Total high-risk customers (score > 70): {len(high_risk):,}")


> **Finding (Q6 answered):** The RE-ENGAGE segment has the highest churn risk scores, as expected. But some customers in RETAIN and NURTURE also have elevated scores. The churn score table lets the business prioritise who to contact first — starting with RE-ENGAGE customers who also had high past spend.


### Top At-Risk Customers — Export for Win-Back Campaign


In [ ]:
at_risk = (
    all_segments[all_segments['Segment'] == 'RE-ENGAGE']
    .sort_values('ChurnScore', ascending=False)
    .head(20)
    .reset_index(drop=True)
    [['Customer ID', 'Recency', 'Frequency', 'MonetaryValue', 'ChurnScore']]
)

at_risk.columns = ['Customer ID', 'Days Since Purchase', 'Total Orders', 'Total Spend (£)', 'Churn Score']
at_risk


---
## Step 17 — Business Recommendations

### Final Segment Overview


In [ ]:
final_summary = all_segments.groupby('Segment').agg(
    Customers=('Customer ID', 'count'),
    Avg_Recency=('Recency', 'mean'),
    Avg_Orders=('Frequency', 'mean'),
    Avg_Spend=('MonetaryValue', 'mean'),
    Avg_ChurnRisk=('ChurnScore', 'mean')
).round(1).sort_values('Avg_Spend', ascending=False)

final_summary


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('All 7 Segments — Final Overview', fontweight='bold')

seg_colors = {
    'DELIGHT': '#e377c2', 'PAMPER': '#9467bd', 'UPSELL': '#8c564b',
    'REWARD': '#d62728', 'RETAIN': '#1f77b4', 'NURTURE': '#2ca02c',
    'RE-ENGAGE': '#ff7f0e'
}
plot_order = final_summary.index.tolist()
bar_colors = [seg_colors[s] for s in plot_order]

# Customer count
final_summary['Customers'].plot(kind='bar', ax=axes[0], color=bar_colors, edgecolor='white')
axes[0].set_title('Customers per Segment')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35)

# Avg spend
final_summary['Avg_Spend'].plot(kind='bar', ax=axes[1], color=bar_colors, edgecolor='white')
axes[1].set_title('Average Total Spend (£)')
axes[1].set_ylabel('£')
axes[1].tick_params(axis='x', rotation=35)

# Avg churn risk
final_summary['Avg_ChurnRisk'].plot(kind='bar', ax=axes[2], color=bar_colors, edgecolor='white')
axes[2].set_title('Average Churn Risk Score')
axes[2].set_ylabel('Score (0–100)')
axes[2].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.show()


In [ ]:
# Normalised heatmap — one glance to compare all segments
heat_data = all_segments.groupby('Segment')[['Recency', 'Frequency', 'MonetaryValue']].mean()
heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min())

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heat_norm,
    annot=heat_data.round(0), fmt='.0f',
    cmap='RdYlGn_r',
    linewidths=0.5,
    ax=ax
)
ax.set_title('Segment RFM Heatmap
(colour = normalised score, number = actual mean)', fontweight='bold')
plt.tight_layout()
plt.show()


### What the Marketing Team Should Do

| Segment | Action | Why |
|---------|--------|-----|
| **DELIGHT** | VIP programme, dedicated contact, early access | Highest value — losing one is very costly |
| **PAMPER** | Exclusive bundles, premium product offers | Big spenders who can be encouraged to buy more often |
| **UPSELL** | Bigger basket promotions, cross-sell higher-margin products | Buy often but spend less per visit |
| **REWARD** | Points scheme, referral bonuses, thank-you offers | Active and loyal — make them feel valued |
| **RETAIN** | Regular newsletters, seasonal promotions | Stable base — needs gentle engagement to stay active |
| **NURTURE** | Welcome sequence, first-repeat-purchase discount | New or infrequent — habit not yet formed |
| **RE-ENGAGE** | Win-back campaign, "We miss you" offer, survey | Lapsed — last chance before they are gone permanently |

### Budget Allocation Suggestion

| Group | Segments | Share |
|-------|---------|-------|
| High priority (keep and grow) | DELIGHT + PAMPER + REWARD | 50% |
| Medium priority (develop) | UPSELL + RETAIN | 30% |
| Low priority (recover) | NURTURE + RE-ENGAGE | 20% |


---
## Step 18 — Conclusion

### Project Summary

| Item | Result |
|------|--------|
| Raw transactions | 525,461 |
| After cleaning | ~406,000 |
| Unique customers segmented | ~4,300 |
| Core K-Means clusters | 4 |
| Premium outlier segments | 3 |
| **Total named segments** | **7** |
| Algorithm | K-Means (k = 4) |
| Extra features created | ChurnScore |

### What We Did — Step by Step

1. **Understood the business problem** — the retailer treats all customers the same and has no visibility into customer value or churn risk
2. **Explored the data** — found geographic concentration in the UK, seasonal revenue peaks, and a skewed distribution of customer value
3. **Cleaned the data** — removed cancellations, returns, guest checkouts, and admin entries
4. **Built RFM features** — aggregated transaction data into three meaningful customer-level metrics
5. **Fixed the skew** — applied log transformation to make K-Means work properly
6. **Handled outliers** — separated premium VIP customers instead of dropping them
7. **Scaled the features** — put R, F, M on equal footing before clustering
8. **Found the right K** — used the elbow method and silhouette scores to confirm k=4
9. **Validated stability** — confirmed clusters are consistent across random starts
10. **Profiled and labelled** — turned cluster numbers into business-friendly names
11. **Added churn scoring** — gave the marketing team a prioritised at-risk customer list

### Business Value

> If the business contacts the top 50 at-risk RE-ENGAGE customers with a 20% discount offer, and just 20% respond and make one more purchase at an average order value of £500, that is **£5,000 in recovered revenue** from a single targeted campaign.

> Contrast that with sending the same offer to all 4,300 customers — lower response rate, higher cost, and the best customers get an unnecessary discount they were never going to leave anyway.

---

*Dataset: UCI Online Retail II · Tools: Python, Pandas, Scikit-Learn, Seaborn, Matplotlib · Author: Krupal Gohil*
